In [ ]:
import yaml

import pyine.data.traces.dataset_reader
import pyine.data.traces.dataset_utils
import pyine.data.utils.splits

dataset_paths = pyine.data.traces.dataset_utils.get_matching_dataset_paths(
    source_dataset_name="TACO",
    pattern="v1.2/10s10t.*of000026.*.lmdb",
)
split = pyine.data.utils.splits.get_dataset_split_result("TACO")
part_file_paths = pyine.data.utils.splits.get_dataset_split_part_file_paths("TACO")

assert len(part_file_paths) == 26, "unexpected number of split parts?"
assert len(dataset_paths) == len(part_file_paths), "unexpected number of dataset parts?"

all_problem_ids: list[str] = []
part_problem_ids: list[list[str]] = []
for part_file_path in part_file_paths:
    with part_file_path.open("r") as fd:
        target_problem_ids = yaml.safe_load(fd)
    part_problem_ids.append(target_problem_ids)
    assert not any([pid in all_problem_ids for pid in target_problem_ids]), "problem id already seen?"
    all_problem_ids.extend(target_problem_ids)

print(f"dataset metadata parsed: expecting {len(all_problem_ids)} problems over {len(dataset_paths)} parts")
assert set(all_problem_ids) == set(split.identifiers), "unexpected problem ids found?"
assert set(all_problem_ids) == set(split.subset_assignments.keys())

In [ ]:
traced_problem_ids: list[str] = []
for dataset_path, expected_problem_ids in zip(dataset_paths, part_problem_ids):
    print(f"validating: {dataset_path}")
    try:
        reader = pyine.data.traces.dataset_reader.DatasetReader(dataset_path)
        found_problem_ids = [problem_id for problem_id in expected_problem_ids if problem_id in reader.problem_keys]
        print(f"\tfound problems ratio: {len(found_problem_ids) / len(expected_problem_ids):.2f}")
        unexpected_problem_ids = [
            problem_id for problem_id in reader.problem_keys if problem_id not in expected_problem_ids
        ]
        print(f"\tunexpected problems: {len(unexpected_problem_ids)}")
        assert not any([pid in traced_problem_ids for pid in found_problem_ids]), "problem id already seen?"
        traced_problem_ids.extend(found_problem_ids)
    except Exception as e:
        print(f"\tfailed: {e}")

assert len(traced_problem_ids) == len(set(traced_problem_ids)), "duplicate problem ids found?"
assert set(traced_problem_ids) == set(all_problem_ids), "unexpected problem ids found?"

In [ ]:
import pathlib

import orjson

node1_split_path_file = pathlib.Path("data/splits/cluster-splits-1/TACO-split.bin")
node2_split_path_file = pathlib.Path("data/splits/cluster-splits-2/TACO-split.bin")

node1_split_result_dict = orjson.loads(node1_split_path_file.open("rb").read())
node2_split_result_dict = orjson.loads(node2_split_path_file.open("rb").read())
node1_split_result = pyine.data.utils.splits.SplitResult.model_validate(node1_split_result_dict)
node2_split_result = pyine.data.utils.splits.SplitResult.model_validate(node2_split_result_dict)

assert node1_split_result.source_dataset_name == node2_split_result.source_dataset_name
assert node2_split_result.config.model_dump() == node1_split_result.config.model_dump()
assert node1_split_result.source_dataset_hash == node2_split_result.source_dataset_hash
assert node1_split_result.identifiers == node2_split_result.identifiers
assert node1_split_result.subset_assignments == node2_split_result.subset_assignments